In [1]:
# ============================================================================
# CORRUPTION-FAMILY ΔIoU TABLE  (TrashNet)
# ----------------------------------------------------------------------------
# Loads vanilla U²-Net + bg-robust v3 + bg-aug checkpoints, evaluates all three
# on the same 11 background corruptions using the SAME corruption functions
# from the bg-robust v3 training script (BG_FUNCS). Produces:
#   - IoU per (model, corruption)
#   - ΔIoU vs. vanilla
#   - A CSV and a JSON for the paper
#
# Corruption functions are copied VERBATIM from the bg-robust v3 training
# script so that the numbers are directly comparable.
# ============================================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"

import json, math, random, warnings, importlib.util, urllib.request
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)

# ============================================================
# 0. Config
# ============================================================
CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
IMG_EXTS = {".jpg", ".jpeg", ".png"}
MASK_SUFFIX = "_mask.png"

IMG_TRAIN = 320
BATCH = 4
SEED = 42
VAL_FRAC = 0.15
FG_MIN = 0.02
FG_MAX = 0.95
MIN_PER_CLASS = 50
TOL = 10

THRESH_GRID = np.linspace(0.05, 0.95, 19)

# All corruptions we will evaluate on.  These are exactly the names in BG_FUNCS
# below.  "solid" and "color_jitter" are training-only augmentations and are
# omitted from the evaluation table by default.
EVAL_CORRUPTIONS = [
    "noise", "texture", "saltpepper", "gaussian", "perlin",
    "checker", "gradient", "blur", "jpeg", "gray", "speckle",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
    print("Deterministic algorithms: enabled (warn_only=True)")
except Exception as e:
    print("Deterministic algorithms not fully available:", e)

OUT_DIR = "/kaggle/working/corruption_table"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 0b. Paths to checkpoints
# ============================================================
VANILLA_CKPT = "/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/vanilla_v2.pth"
VANILLA_ARCH = "/kaggle/input/notebooks/sumn2u/vanilla-u2-net/vanilla_v2/u2net_arch.py"

BG_ROBUST_CKPT = "/kaggle/input/notebooks/sumn2u/background-robust/bg_robust_v3/bg_robust_v3.pth"

BG_AUG_CKPT = "/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/vanilla_bgaug.pth"
BG_AUG_ARCH = "/kaggle/input/notebooks/sumn2u/vanilla-u-net-with-background-augmentation/vanilla_bgaug/u2net_arch.py"

U2NET_MEAN = np.array([0.485, 0.456, 0.406])
U2NET_STD  = np.array([0.229, 0.224, 0.225])
norm_tf = transforms.Normalize(U2NET_MEAN.tolist(), U2NET_STD.tolist())

# ============================================================
# 1. Locate TrashNet
# ============================================================
def find_root(base, required):
    if not os.path.isdir(base): return None
    if all(os.path.isdir(os.path.join(base, c)) for c in required):
        return base
    for sub in sorted(os.listdir(base)):
        cand = os.path.join(base, sub)
        if os.path.isdir(cand) and all(
            os.path.isdir(os.path.join(cand, c)) for c in required):
            return cand
    return None

DATA_ROOT = None
for c in [
    "/kaggle/input/trashnet-segmentation",
    "/kaggle/input/datasets/sumn2u/trashnet-segmentation",
    "/kaggle/input/trashnet-segmentation/trashnet-segmentation",
]:
    r = find_root(c, CLASSES)
    if r:
        DATA_ROOT = r
        break
assert DATA_ROOT, "Could not locate TrashNet class folders"
print("Dataset root:", DATA_ROOT)

# ============================================================
# 2. Build the same pairs + split as the training scripts
# ============================================================
def rgb_mask_to_binary(mask_path, tol=TOL):
    m = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if m is None: raise IOError(mask_path)
    if m.ndim == 2: return (m > 127).astype(np.uint8) * 255
    if m.shape[2] == 4: return (m[..., 3] > 127).astype(np.uint8) * 255
    diff = m[..., :3].astype(np.int16).sum(axis=2)
    return (diff > tol).astype(np.uint8) * 255

pairs = []
for cls in CLASSES:
    cp = os.path.join(DATA_ROOT, cls)
    if not os.path.isdir(cp): continue
    for f in sorted(os.listdir(cp)):
        if not f.lower().endswith(MASK_SUFFIX): continue
        stem = f[:-len(MASK_SUFFIX)]
        img = None
        for ext in IMG_EXTS:
            cand = os.path.join(cp, stem + ext)
            if os.path.isfile(cand): img = cand; break
        if img is None: continue
        try:
            b = rgb_mask_to_binary(os.path.join(cp, f))
        except Exception:
            continue
        fg = (b > 127).mean()
        if not (FG_MIN < fg < FG_MAX): continue
        pairs.append((img, b.copy(), cls))

counts = pd.Series([p[2] for p in pairs]).value_counts()
CLASSES_TRAINING = [c for c in CLASSES if counts.get(c, 0) >= MIN_PER_CLASS]
pairs = [p for p in pairs if p[2] in CLASSES_TRAINING]

random.seed(SEED); random.shuffle(pairs)
by_class = defaultdict(list)
for p in pairs: by_class[p[2]].append(p)
train_pairs, val_pairs = [], []
for cls, items in by_class.items():
    n_val = max(1, int(round(VAL_FRAC * len(items))))
    val_pairs.extend(items[:n_val])
    train_pairs.extend(items[n_val:])
random.shuffle(train_pairs); random.shuffle(val_pairs)
print(f"Train: {len(train_pairs)}   Val: {len(val_pairs)}")
print("Classes used:", CLASSES_TRAINING)

# ============================================================
# 3. Corruption library — COPIED VERBATIM from bg-robust v3 training
# ============================================================
def _to_u8(x): return np.clip(x, 0, 255).astype(np.uint8)

def _bg_gray(img, bg, rng, **kw):
    out = img.copy(); out[bg] = 128; return out

def _bg_noise(img, bg, rng, **kw):
    amp = rng.uniform(0.10, 1.0)
    n = (rng.randint(0, 256, size=img.shape) * amp).astype(np.uint8)
    out = img.copy(); out[bg] = n[bg]; return out

def _bg_gaussian(img, bg, rng, **kw):
    sig = rng.uniform(5, 80)
    n = rng.normal(0, sig, img.shape).astype(np.int16)
    return _to_u8(img.astype(np.int16) + n)

def _bg_speckle(img, bg, rng, **kw):
    s = rng.uniform(0.05, 0.80)
    n = rng.normal(0, s, img.shape)
    return _to_u8(img.astype(np.float32) * (1 + n))

def _bg_saltpepper(img, bg, rng, **kw):
    p = rng.uniform(0.02, 0.25)
    sp = img.copy()
    m = rng.rand(*img.shape[:2]) < p
    sp[m] = (rng.randint(0, 2, size=(int(m.sum()), 3)) * 255).astype(np.uint8)
    return sp

def _bg_blur(img, bg, rng, **kw):
    k = int(rng.choice([3, 7, 15, 31, 63]))
    out = img.copy(); out[bg] = cv2.GaussianBlur(img, (k, k), 0)[bg]; return out

def _bg_checker(img, bg, rng, **kw):
    tile = int(rng.choice([2, 4, 8, 16, 32]))
    H, W = img.shape[:2]
    pat = ((np.indices((H, W)).sum(0) // tile) % 2 * 255).astype(np.uint8)
    pat3 = np.stack([pat]*3, -1)
    out = img.copy(); out[bg] = pat3[bg]; return out

def _bg_gradient(img, bg, rng, **kw):
    H, W = img.shape[:2]
    lo = rng.randint(0, 128); hi = rng.randint(128, 256)
    g = np.linspace(lo, hi, W).astype(np.uint8)
    g3 = np.stack([np.tile(g, (H, 1))]*3, -1)
    out = img.copy(); out[bg] = g3[bg]; return out

def _bg_perlin(img, bg, rng, **kw):
    H, W = img.shape[:2]
    scale = int(rng.choice([8, 16, 32, 64]))
    small = rng.rand(max(2, H // scale), max(2, W // scale)).astype(np.float32)
    up = cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC)
    up = (up - up.min()) / max(1e-6, up.max() - up.min())
    up = (up * 255).astype(np.uint8)
    out = img.copy(); out[bg] = np.stack([up]*3, -1)[bg]; return out

def _bg_texture(img, bg, rng, **kw):
    H, W = img.shape[:2]
    scale = int(rng.choice([4, 8, 16, 32, 64]))
    t = rng.randint(0, 256, size=(H // scale + 1, W // scale + 1, 3),
                    dtype=np.uint8)
    t = cv2.resize(t, (W, H), interpolation=cv2.INTER_NEAREST)
    out = img.copy(); out[bg] = t[bg]; return out

def _bg_jpeg(img, bg, rng, **kw):
    q = int(rng.uniform(5, 60))
    enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), q])[1]
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    out = img.copy(); out[bg] = dec[bg]; return out

def _bg_solid(img, bg, rng, **kw):
    c = rng.randint(0, 256, 3, dtype=np.uint8)
    out = img.copy(); out[bg] = c; return out

def _bg_color_jitter(img, bg, rng, **kw):
    shift = rng.randint(-80, 80, 3).astype(np.int16)
    out = np.clip(img.astype(np.int16) + shift, 0, 255).astype(np.uint8)
    out[bg] = out[bg]
    return out

BG_FUNCS = {
    "gray":         _bg_gray,
    "noise":        _bg_noise,
    "gaussian":     _bg_gaussian,
    "speckle":      _bg_speckle,
    "saltpepper":   _bg_saltpepper,
    "blur":         _bg_blur,
    "checker":      _bg_checker,
    "gradient":     _bg_gradient,
    "perlin":       _bg_perlin,
    "texture":      _bg_texture,
    "jpeg":         _bg_jpeg,
    "solid":        _bg_solid,
    "color_jitter": _bg_color_jitter,
}

def _randomize_background(img, msk, mode=None, rng=None):
    assert rng is not None, "pass an explicit np.random.RandomState"
    if mode is None:
        # only used during training; not needed here, but kept for parity
        mode = rng.choice(list(BG_FUNCS.keys()))
    fg = (msk > 127); bg = ~fg
    return BG_FUNCS[mode](img, bg, rng)

# ============================================================
# 4. Eval dataset — one variant per corruption, same seed as training
# ============================================================
class EvalDS(Dataset):
    def __init__(self, pairs, size=IMG_TRAIN, bg_mode=None):
        self.pairs = pairs; self.size = size; self.bg_mode = bg_mode
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, bin_mask, _ = self.pairs[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.size, self.size))
        msk = cv2.resize(bin_mask, (self.size, self.size),
                         interpolation=cv2.INTER_NEAREST)
        if self.bg_mode is not None:
            rng = np.random.RandomState(SEED + i)   # same seed as training
            img = _randomize_background(img, msk, mode=self.bg_mode, rng=rng)
        x = norm_tf(transforms.ToTensor()(Image.fromarray(img)))
        y = (torch.from_numpy(msk).float().unsqueeze(0) > 127).float()
        return x, y

def make_loader(ds, shuffle=False):
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                      num_workers=2, pin_memory=True, generator=g)

val_loader_clean = make_loader(EvalDS(val_pairs))
val_loaders = {c: make_loader(EvalDS(val_pairs, bg_mode=c))
               for c in EVAL_CORRUPTIONS}

# ============================================================
# 5. Architecture — vanilla U²-Net (from arch file) + bg-robust (inlined)
# ============================================================
def load_arch(arch_path):
    spec = importlib.util.spec_from_file_location("u2net_arch", arch_path)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod.U2NET

def _upsample_like(x, ref):
    size = tuple(ref.shape[-2:]) if torch.is_tensor(ref) else tuple(ref)
    return nn.Upsample(size=size, mode='bilinear', align_corners=False)(x)

def _size_map(x, height):
    size = list(x.shape[-2:]); sizes = {}
    for h in range(1, height):
        sizes[h] = size
        size = [math.ceil(w / 2) for w in size]
    return sizes

class REBNCONV(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, dilate=1):
        super().__init__()
        self.conv_s1 = nn.Conv2d(in_ch, out_ch, 3, padding=dilate, dilation=dilate)
        self.bn_s1 = nn.BatchNorm2d(out_ch); self.relu_s1 = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu_s1(self.bn_s1(self.conv_s1(x)))

class MultiScaleExpert(nn.Module):
    def __init__(self, in_ch, out_ch, dilations=(1, 2, 4)):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=d, dilation=d, bias=False),
                          nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
            for d in dilations])
        self.gate = nn.Conv2d(in_ch, len(dilations), 1)
    def forward(self, x):
        outs = [e(x) for e in self.experts]
        stacked = torch.stack(outs, dim=1)
        gate_w = F.softmax(self.gate(x), dim=1).unsqueeze(2)
        return (stacked * gate_w).sum(dim=1)

class RSU(nn.Module):
    def __init__(self, name, height, in_ch, mid_ch, out_ch, use_moe=False):
        super().__init__()
        self.name = name; self.height = height; self.use_moe = use_moe
        self._make_layers(in_ch, mid_ch, out_ch, use_moe)
    def forward(self, x):
        sizes = _size_map(x, self.height)
        x0 = self.rebnconvin(x)
        def unet(x, height=1):
            if height < self.height:
                x1 = getattr(self, f'rebnconv{height}')(x)
                if height < self.height - 1:
                    x2 = unet(getattr(self, 'downsample')(x1), height + 1)
                else:
                    x2 = self.moe(x1) if self.use_moe else x1
                x = getattr(self, f'rebnconv{height}d')(torch.cat((x2, x1), 1))
                return _upsample_like(x, sizes[height - 1]) if height > 1 else x
            return getattr(self, f'rebnconv{height}')(x)
        return x0 + unet(x0)
    def _make_layers(self, in_ch, mid_ch, out_ch, use_moe):
        self.add_module('rebnconvin', REBNCONV(in_ch, out_ch))
        self.add_module('downsample', nn.MaxPool2d(2, stride=2, ceil_mode=True))
        self.add_module('rebnconv1', REBNCONV(out_ch, mid_ch))
        self.add_module('rebnconv1d', REBNCONV(mid_ch*2, out_ch))
        for i in range(2, self.height):
            self.add_module(f'rebnconv{i}', REBNCONV(mid_ch, mid_ch))
            self.add_module(f'rebnconv{i}d', REBNCONV(mid_ch*2, mid_ch))
        self.add_module(f'rebnconv{self.height}', REBNCONV(mid_ch, mid_ch))
        if use_moe:
            self.moe = MultiScaleExpert(mid_ch, mid_ch, dilations=(1, 2, 4))

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True),
                                  nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.psi(self.relu(self.W_g(g) + self.W_x(x)))
        return x * psi

class FactorizedHead(nn.Module):
    def __init__(self, in_ch=64, mid_ch=64):
        super().__init__()
        self.object_branch = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, mid_ch, 3, padding=1, bias=False))
        self.context_branch = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, mid_ch, 3, padding=1, bias=False))
    def forward(self, feat): return self.object_branch(feat), self.context_branch(feat)

class BackgroundRobustU2Net(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, n_classes=6):
        super().__init__()
        self.stage1 = RSU('stage1', 7, in_ch, 32, 64); self.pool12 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage2 = RSU('stage2', 6, 64, 32, 128);   self.pool23 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage3 = RSU('stage3', 5, 128, 64, 256);  self.pool34 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage4 = RSU('stage4', 4, 256, 128, 512); self.pool45 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage5 = RSU('stage5', 4, 512, 256, 512, use_moe=True)
        self.pool56 = nn.MaxPool2d(2, 2, ceil_mode=True)
        self.stage6 = RSU('stage6', 4, 512, 256, 512, use_moe=True)
        self.stage5d = RSU('stage5d', 4, 1024, 256, 512)
        self.stage4d = RSU('stage4d', 4, 1024, 128, 256)
        self.stage3d = RSU('stage3d', 5, 512, 64, 128)
        self.stage2d = RSU('stage2d', 6, 256, 32, 64)
        self.stage1d = RSU('stage1d', 7, 128, 16, 64)
        self.att5 = AttentionGate(512, 512, 256)
        self.att4 = AttentionGate(512, 512, 256)
        self.att3 = AttentionGate(256, 256, 128)
        self.factor_head = FactorizedHead(64, 64)
        self.fg_frac_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                           nn.Linear(512, 64), nn.ReLU(inplace=True),
                                           nn.Linear(64, 1), nn.Sigmoid())
        self.ctx_classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                             nn.Linear(64, 64), nn.ReLU(inplace=True),
                                             nn.Linear(64, n_classes))
        self.side1 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side2 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side3 = nn.Conv2d(128, out_ch, 3, padding=1)
        self.side4 = nn.Conv2d(256, out_ch, 3, padding=1)
        self.side5 = nn.Conv2d(512, out_ch, 3, padding=1)
        self.side6 = nn.Conv2d(512, out_ch, 3, padding=1)
        self.outconv = nn.Conv2d(6 * out_ch, out_ch, 1)
    def forward(self, x, return_aux=False):
        hx1 = self.stage1(x); hx = self.pool12(hx1)
        hx2 = self.stage2(hx); hx = self.pool23(hx2)
        hx3 = self.stage3(hx); hx = self.pool34(hx3)
        hx4 = self.stage4(hx); hx = self.pool45(hx4)
        hx5 = self.stage5(hx); hx = self.pool56(hx5)
        hx6 = self.stage6(hx)
        hx6up = _upsample_like(hx6, hx5)
        hx5d = self.stage5d(torch.cat((hx6up, self.att5(g=hx6up, x=hx5)), 1))
        hx5dup = _upsample_like(hx5d, hx4)
        hx4d = self.stage4d(torch.cat((hx5dup, self.att4(g=hx5dup, x=hx4)), 1))
        hx4dup = _upsample_like(hx4d, hx3)
        hx3d = self.stage3d(torch.cat((hx4dup, self.att3(g=hx4dup, x=hx3)), 1))
        hx3dup = _upsample_like(hx3d, hx2)
        hx2d = self.stage2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)
        hx1d = self.stage1d(torch.cat((hx2dup, hx1), 1))
        obj_feat, ctx_feat = self.factor_head(hx1d)
        d1 = self.side1(obj_feat)
        d2 = _upsample_like(self.side2(hx2d), d1)
        d3 = _upsample_like(self.side3(hx3d), d1)
        d4 = _upsample_like(self.side4(hx4d), d1)
        d5 = _upsample_like(self.side5(hx5d), d1)
        d6 = _upsample_like(self.side6(hx6), d1)
        d0 = self.outconv(torch.cat((d1, d2, d3, d4, d5, d6), 1))
        return d0, d1, d2, d3, d4, d5, d6

# ============================================================
# 6. Model loaders
# ============================================================
def load_vanilla(ckpt_path, arch_path, label):
    BaseU2NET = load_arch(arch_path)
    model = BaseU2NET().to(DEVICE)
    sd = torch.load(ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd: sd = sd["state_dict"]
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"[{label}] missing={len(missing)}  unexpected={len(unexpected)}")
    model.eval(); return model

def load_bg_robust(ckpt_path, n_classes=6, label="bg_robust"):
    model = BackgroundRobustU2Net(n_classes=n_classes).to(DEVICE)
    sd = torch.load(ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd: sd = sd["state_dict"]
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"[{label}] missing={len(missing)}  unexpected={len(unexpected)}")
    model.eval(); return model

# ============================================================
# 7. Evaluation
# ============================================================
@torch.no_grad()
def eval_iou(model, loader, thr=0.5):
    model.eval(); ious = []
    for x, y in loader:
        x = x.to(DEVICE)
        d0, *_ = model(x)
        p = (torch.sigmoid(d0) > thr).float().cpu()
        y = y.cpu()
        for k in range(p.shape[0]):
            pk, yk = p[k, 0], y[k, 0]
            inter = (pk * yk).sum().item()
            union = ((pk + yk) > 0).sum().item()
            ious.append(inter / union if union > 0 else 0.0)
    return float(np.mean(ious))

def best_threshold_on_clean(model):
    ious = [eval_iou(model, val_loader_clean, thr=float(t)) for t in THRESH_GRID]
    idx = int(np.argmax(ious))
    return float(THRESH_GRID[idx]), float(ious[idx])

# ============================================================
# 8. Run
# ============================================================
print("\n" + "=" * 70)
print("CORRUPTION-FAMILY ΔIoU TABLE")
print("=" * 70)

models = {}
models["vanilla"]   = load_vanilla(VANILLA_CKPT, VANILLA_ARCH, "vanilla")
models["bg_robust"] = load_bg_robust(BG_ROBUST_CKPT, n_classes=len(CLASSES_TRAINING), label="bg_robust")
models["bg_aug"]    = load_vanilla(BG_AUG_CKPT, BG_AUG_ARCH, "bg_aug")

results = {}
for name, model in models.items():
    thr, clean_iou = best_threshold_on_clean(model)
    print(f"\n[{name}] best threshold on clean = {thr:.2f}  clean IoU = {clean_iou:.4f}")
    ious = {"clean": clean_iou}
    for c in EVAL_CORRUPTIONS:
        iou = eval_iou(model, val_loaders[c], thr=thr)
        ious[c] = iou
        print(f"    {c:<12} : {iou:.4f}")
    results[name] = {"threshold": thr, "ious": ious}

# ============================================================
# 9. ΔIoU table vs vanilla
# ============================================================
print("\n" + "=" * 70)
print("ΔIoU vs. VANILLA")
print("=" * 70)

rows = []
for c in EVAL_CORRUPTIONS:
    v  = results["vanilla"]["ious"][c]
    br = results["bg_robust"]["ious"][c]
    ba = results["bg_aug"]["ious"][c]
    rows.append({
        "corruption": c,
        "vanilla":    round(v, 4),
        "bg_robust":  round(br, 4),
        "bg_aug":     round(ba, 4),
        "ΔIoU_robust_vs_vanilla": round(br - v, 4),
        "ΔIoU_bgaug_vs_vanilla":  round(ba - v, 4),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(os.path.join(OUT_DIR, "corruption_table.csv"), index=False)

# ============================================================
# 10. Save JSON
# ============================================================
out = {
    "config": {
        "seed": SEED,
        "img_size": IMG_TRAIN,
        "n_train": len(train_pairs),
        "n_val": len(val_pairs),
        "classes": CLASSES_TRAINING,
        "corruptions": EVAL_CORRUPTIONS,
        "checkpoints": {
            "vanilla":   VANILLA_CKPT,
            "bg_robust": BG_ROBUST_CKPT,
            "bg_aug":    BG_AUG_CKPT,
        },
    },
    "results": results,
    "delta_vs_vanilla": rows,
}
with open(os.path.join(OUT_DIR, "corruption_table.json"), "w") as f:
    json.dump(out, f, indent=2, default=float)
print(f"\nSaved -> {os.path.join(OUT_DIR, 'corruption_table.csv')}")
print(f"Saved -> {os.path.join(OUT_DIR, 'corruption_table.json')}")

# ============================================================
# 11. Bar chart
# ============================================================
try:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(EVAL_CORRUPTIONS))
    w = 0.27
    v_vals = [results["vanilla"]["ious"][c]   for c in EVAL_CORRUPTIONS]
    b_vals = [results["bg_robust"]["ious"][c] for c in EVAL_CORRUPTIONS]
    a_vals = [results["bg_aug"]["ious"][c]    for c in EVAL_CORRUPTIONS]
    ax.bar(x - w,   v_vals, w, label="vanilla")
    ax.bar(x,       b_vals, w, label="bg_robust")
    ax.bar(x + w,   a_vals, w, label="bg_aug")
    ax.set_xticks(x); ax.set_xticklabels(EVAL_CORRUPTIONS, rotation=45, ha="right")
    ax.set_ylabel("IoU"); ax.set_title("IoU by corruption")
    ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "corruption_table.png"), dpi=130); plt.close()
    print(f"Saved -> {os.path.join(OUT_DIR, 'corruption_table.png')}")
except Exception as e:
    print("Plot failed:", e)

print("\nDone.")

Device: cuda
Deterministic algorithms: enabled (warn_only=True)
Dataset root: /kaggle/input/datasets/sumn2u/trashnet-segmentation
Train: 1902   Val: 334
Classes used: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

CORRUPTION-FAMILY ΔIoU TABLE
[vanilla] missing=0  unexpected=0
[bg_robust] missing=0  unexpected=0
[bg_aug] missing=0  unexpected=0

[vanilla] best threshold on clean = 0.55  clean IoU = 0.9685
    noise        : 0.6988
    texture      : 0.4223
    saltpepper   : 0.4104
    gaussian     : 0.6325
    perlin       : 0.8921
    checker      : 0.4076
    gradient     : 0.9861
    blur         : 0.9743
    jpeg         : 0.9563
    gray         : 0.9878
    speckle      : 0.5283

[bg_robust] best threshold on clean = 0.45  clean IoU = 0.9622
    noise        : 0.9836
    texture      : 0.9718
    saltpepper   : 0.8029
    gaussian     : 0.8534
    perlin       : 0.9794
    checker      : 0.4139
    gradient     : 0.9863
    blur         : 0.9673
    jpeg         : 